In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import datetime
import time

url = "https://bank.gov.ua/ua/markets/exchangerates"
custom_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
}
today = datetime.date.today()
start = today.replace(day=1)
rates_data = {"date": [], "USD": [], "EUR": []}
current = start

while current <= today:
    custom_params = {
        "date": current.strftime("%d.%m.%Y"),
        "period": "daily"
    }
    result = requests.get(url, headers=custom_headers, params=custom_params)
    soup = BeautifulSoup(result.text, "html.parser")
    table = soup.find("table", attrs={"id": "exchangeRates"})
    rows = table.find("tbody").find_all("tr")

    day_rates = {}
    for row in rows:
        cells = row.find_all("td")
        code = cells[1].text.strip()
        rate = cells[4].text.strip()
        if code in ("USD", "EUR"):
            day_rates[code] = float(rate.replace(",", "."))

    rates_data["date"].append(current)
    rates_data["USD"].append(day_rates.get("USD"))
    rates_data["EUR"].append(day_rates.get("EUR"))

    current += datetime.timedelta(days=1)
    time.sleep(0.3)

rates_df = pd.DataFrame(rates_data)
rates_df

,date,USD,EUR
0,2026-09-01,44.5249,51.6369
1,2026-09-02,44.4553,51.5357
2,2026-09-03,44.6093,51.6433
3,2026-09-04,44.7273,51.9382
4,2026-09-05,44.7273,51.9382
5,2026-09-06,44.7273,51.9382
6,2026-09-07,44.5616,51.7882
7,2026-09-08,44.4654,51.6817
8,2026-09-09,44.4694,51.6383
9,2026-09-10,44.6462,51.9905


In [2]:
rates_df["EUR_to_USD"] = rates_df["EUR"] / rates_df["USD"]
rates_df

,date,USD,EUR,EUR_to_USD
0,2026-09-01,44.5249,51.6369,1.159731
1,2026-09-02,44.4553,51.5357,1.159270
2,2026-09-03,44.6093,51.6433,1.157680
3,2026-09-04,44.7273,51.9382,1.161219
4,2026-09-05,44.7273,51.9382,1.161219
5,2026-09-06,44.7273,51.9382,1.161219
6,2026-09-07,44.5616,51.7882,1.162171
7,2026-09-08,44.4654,51.6817,1.162290
8,2026-09-09,44.4694,51.6383,1.161210
9,2026-09-10,44.6462,51.9905,1.164500


In [3]:
import plotly.express as px

fig = px.line(
    rates_df,
    x="date",
    y="EUR_to_USD",
    markers=True,
    title="Соотношение курса EUR к USD по дням месяца",
    labels={"date": "Дата", "EUR_to_USD": "EUR / USD"}
)
fig.show()

In [4]:
sites = [
    "https://bank.gov.ua",
    "https://dou.ua",
    "https://rozetka.com.ua",
    "https://privatbank.ua",
    "https://olx.ua",
    "https://this-site-does-not-exist-12345.ua",
    "https://bank.gov.ua/nonexistent-page-404",
]
for site in sites:
    try:
        result = requests.get(site, headers=custom_headers, timeout=5)
        if result.status_code == 200:
            print(f"{site} — доступен, код {result.status_code}")
        elif 400 <= result.status_code < 500:
            print(f"{site} — сервер отвечает, но страница недоступна, код {result.status_code}")
        elif result.status_code >= 500:
            print(f"{site} — сервер отвечает с ошибкой, код {result.status_code}")
        else:
            print(f"{site} — доступен, код {result.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"{site} — недоступен, ошибка: {e}")

https://bank.gov.ua — доступен, код 200
https://dou.ua — доступен, код 200
https://rozetka.com.ua — сервер отвечает, но страница недоступна, код 403
https://privatbank.ua — доступен, код 200
https://olx.ua — сервер отвечает, но страница недоступна, код 403
https://this-site-does-not-exist-12345.ua — недоступен, ошибка: HTTPSConnectionPool(host='this-site-does-not-exist-12345.ua', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='this-site-does-not-exist-12345.ua', port=443): Failed to resolve 'this-site-does-not-exist-12345.ua' ([Errno 11001] getaddrinfo failed)"))
https://bank.gov.ua/nonexistent-page-404 — сервер отвечает, но страница недоступна, код 404


In [5]:
url = "https://dou.ua"
result = requests.get(url, headers=custom_headers, timeout=10)
soup = BeautifulSoup(result.text, "html.parser")
links = soup.find_all("a", href=True)
fresh_links = []
seen = set()
for link in links:
    href = link["href"]
    if "from=recent" in href:
        clean_href = href.split("?")[0]
        title = link.text.strip().replace("\xa0", " ")
        if clean_href not in seen and title:
            seen.add(clean_href)
            fresh_links.append((title, clean_href))
fresh_df = pd.DataFrame(fresh_links, columns=["title", "url"])
fresh_df.to_csv("data/dou_fresh_articles.csv", sep=",", index=False, encoding="utf-8-sig")
fresh_df

,title,url
0,Після 40+ років роботу шукати складніше — анал...,https://dou.ua/lenta/articles/ageism-in-it-ana...
1,Досвід знецінюється | QA отримують більше | Ма...,https://dou.ua/lenta/articles/analytics-youtub...
2,Чи втрачали айтівці бронювання після аудиту та...,https://dou.ua/lenta/articles/how-the-booking-...
3,"«Дистанційно ми інженерів не навчимо», — як тр...",https://dou.ua/lenta/articles/education-under-...
4,GPT-6 Astra вражає | Що по Starlink для дронів...,https://dou.ua/lenta/articles/dou-news-266/
5,"Що треба айтівцям, аби зберігати оптимізм — ре...",https://dou.ua/lenta/articles/dou-day-picnic-2...
6,Наскільки добре ти працюєш із ШІ? Нові інструм...,https://dou.ua/lenta/articles/ai-tools-3/
7,Чому Київ атакують реактивними шахедами вдень ...,https://dou.ua/lenta/interviews/flash-about-je...
